# Huấn luyện Shopformer trên PoseLift

Notebook này huấn luyện mô hình phát hiện trộm cắp dựa trên khung xương theo hai giai đoạn:

1. **GCAE** (ST-GCN autoencoder) học cách nén chuỗi khung xương thành 2 token.
2. **Transformer** encoder-decoder học tái tạo token, chỉ trên hành vi **bình thường**. Điểm bất thường là sai số tái tạo.

**Hãy đọc trước khi chạy**

- Repo chính thức `TeCSAR-UNCC/Shopformer` hiện **chỉ có README, `environment.yml` và ảnh**, không có mã huấn luyện, và lệnh Stage 1 trong README bị để trống. Vì vậy notebook này là **cài đặt lại theo bài báo và README**, không phải mã chính thức. Kết quả sẽ **không trùng** số trong bài báo (AUC-ROC 69,15 / AUC-PR 44,49 / EER 0,38).
- Những chỗ mình tự chọn vì bài báo không nói rõ: 18 điểm khung xương = COCO17 + điểm cổ (README dùng `num_kp 18`, `8 kênh × 18 = 144`); cách chuẩn hóa tọa độ; tính loss trong không gian khung xương qua decoder đã đóng băng; cách chia train/test.
- README dùng `seg_len 24, seg_stride 12`, trong khi bản HTML của bài báo ghi cửa sổ 12. Notebook theo README.
- Dữ liệu: tải PoseLift từ Google Drive trong repo [PoseLift](https://github.com/TeCSAR-UNCC/PoseLift) về máy, rồi đưa vào Google Drive của bạn (hoặc thư mục cục bộ). Mỗi video có một file `.pkl` (khung xương) và một file `.npy` (nhãn từng khung hình) cùng tên.

In [ ]:
%pip install -q scikit-learn scipy matplotlib

import os, sys, json, pickle, random, re, shutil, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from scipy.ndimage import gaussian_filter1d
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, roc_curve

if not Path("/kaggle/input").exists():  # Chỉ gắn Google Drive khi chạy trên Google Colab thật
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

def _find_default_data_root() -> str:
    for candidate in (
        Path("/kaggle/input/shopfomer/PoseLift/Pickle_files"),
        Path("/kaggle/input/shopfomer/PoseLift"),
        Path("/kaggle/input"),
        Path("/content/drive/MyDrive/PoseLift"),
    ):
        if candidate.exists() and any(candidate.rglob("*.pkl")):
            return str(candidate)
    return "/kaggle/input/shopfomer/PoseLift"

CFG = dict(
    data_root=os.environ.get("SHOPLIFT_DATA", _find_default_data_root()),  # tự nhận /kaggle/input/shopfomer/PoseLift
    out_dir=os.environ.get("SHOPLIFT_OUT", "/kaggle/working/checkpoints/shopformer_lite_v1" if Path("/kaggle/working").exists() else "checkpoints/shopformer_lite_v1"),
    seg_len=24, seg_stride=12, num_kp=18,          # theo README
    tok_ch=8, n_tokens=2,                          # token = 8 kênh x 18 điểm = 144
    nhead=12, num_layers=4, ff_dim=64, dropout=0.1,  # theo README
    gcae_epochs=int(os.environ.get("EP_GCAE", 20)), gcae_lr=1e-3,
    tr_epochs=int(os.environ.get("EP_TR", 30)), tr_lr=1e-4,   # README: 10 epoch, lr 5e-5 (xem ghi chú ở giai đoạn 2)
    batch=128, test_normal_frac=0.2, val_frac=0.1,
    smooth_sigma=8, threshold_pct=99.0, seed=0,
)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
print("thiết bị:", device)

In [ ]:
%%writefile shopformer_lite.py
"""Shopformer-style model: GCAE tokenizer + Transformer encoder-decoder.

Cài đặt lại theo bài báo (arXiv 2504.19970) và README của repo chính thức,
KHÔNG phải mã chính thức. Kiến trúc:
  1) GCAE (ST-GCN autoencoder) nén chuỗi khung xương thành n_tokens token.
  2) Transformer encoder-decoder tái tạo token. Token tái tạo đi qua decoder
     GCAE (đã đóng băng) để ra chuỗi khung xương; sai số MSE là điểm bất thường.
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------------------------------------------------------- skeleton
COCO_EDGES = [(0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9), (6, 8),
              (8, 10), (5, 11), (6, 12), (11, 12), (11, 13), (13, 15),
              (12, 14), (14, 16)]
NECK_EDGES = [(17, 5), (17, 6), (17, 0)]   # điểm 17 = cổ (trung điểm hai vai)


def add_neck(kp: np.ndarray) -> np.ndarray:
    """[..., 17, 2] -> [..., 18, 2] bằng cách thêm điểm cổ."""
    neck = (kp[..., 5:6, :] + kp[..., 6:7, :]) / 2.0
    return np.concatenate([kp, neck], axis=-2)


def skeleton_adjacency(num_kp: int = 18) -> torch.Tensor:
    A = torch.eye(num_kp)
    for i, j in COCO_EDGES + NECK_EDGES:
        if i < num_kp and j < num_kp:
            A[i, j] = A[j, i] = 1.0
    return A / A.sum(1, keepdim=True)


# ---------------------------------------------------------------- GCAE
class STBlock(nn.Module):
    """Một khối ST-GCN: tích chập đồ thị theo không gian + tích chập thời gian."""

    def __init__(self, cin, cout, A, stride=1, kt=3):
        super().__init__()
        self.register_buffer("A", A.clone())
        self.edge_w = nn.Parameter(torch.ones_like(A))
        self.gcn = nn.Conv2d(cin, cout, 1)
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, (kt, 1), (stride, 1), (kt // 2, 0)),
            nn.BatchNorm2d(cout))
        self.res = (nn.Identity() if (cin == cout and stride == 1)
                    else nn.Conv2d(cin, cout, 1, (stride, 1)))
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):                       # x: [B, C, T, K]
        y = torch.einsum("bctk,kv->bctv", self.gcn(x), self.A * self.edge_w)
        return self.act(self.tcn(y) + self.res(x))


class GCAE(nn.Module):
    def __init__(self, num_kp=18, in_ch=2, hid=32, tok_ch=8,
                 seg_len=24, n_tokens=2):
        super().__init__()
        A = skeleton_adjacency(num_kp)
        self.num_kp, self.in_ch, self.tok_ch = num_kp, in_ch, tok_ch
        self.seg_len, self.n_tokens = seg_len, n_tokens
        self.token_dim = tok_ch * num_kp         # 8 * 18 = 144
        self.enc = nn.Sequential(
            STBlock(in_ch, hid, A), STBlock(hid, hid * 2, A, stride=2),
            STBlock(hid * 2, hid * 2, A, stride=2),
            nn.Conv2d(hid * 2, tok_ch, 1))
        self.dec = nn.Sequential(
            STBlock(tok_ch, hid * 2, A), STBlock(hid * 2, hid, A),
            STBlock(hid, hid, A), nn.Conv2d(hid, in_ch, 1))

    def encode(self, x):                         # x: [B, C, T, K] -> [B, n, D]
        z = self.enc(x)
        z = F.adaptive_avg_pool2d(z, (self.n_tokens, self.num_kp))
        b = z.size(0)
        return z.permute(0, 2, 1, 3).reshape(b, self.n_tokens, self.token_dim)

    def decode(self, tokens):                    # [B, n, D] -> [B, C, T, K]
        b = tokens.size(0)
        z = tokens.reshape(b, self.n_tokens, self.tok_ch, self.num_kp)
        z = z.permute(0, 2, 1, 3)
        z = F.interpolate(z, size=(self.seg_len, self.num_kp), mode="bilinear",
                          align_corners=False)
        return self.dec(z)

    def forward(self, x):
        return self.decode(self.encode(x))


# ---------------------------------------------------------------- Transformer
class TokenTransformer(nn.Module):
    def __init__(self, d_model=144, nhead=12, num_layers=4, ff_dim=64,
                 n_tokens=2, dropout=0.1):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, n_tokens, d_model) * 0.02)
        self.query = nn.Parameter(torch.randn(1, n_tokens, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model, nhead, ff_dim, dropout,
                                         batch_first=True)
        dec = nn.TransformerDecoderLayer(d_model, nhead, ff_dim, dropout,
                                         batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, num_layers)
        self.decoder = nn.TransformerDecoder(dec, num_layers)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, tokens):                   # [B, n, D] -> [B, n, D]
        mem = self.encoder(tokens + self.pos)
        q = self.query.expand(tokens.size(0), -1, -1) + self.pos
        return self.out(self.decoder(q, mem))


class Shopformer(nn.Module):
    def __init__(self, gcae: GCAE, transformer: TokenTransformer):
        super().__init__()
        self.gcae, self.transformer = gcae, transformer

    def reconstruct(self, x):
        return self.gcae.decode(self.transformer(self.gcae.encode(x)))

    def forward(self, x):
        return self.reconstruct(x)


def build_models(cfg: dict):
    gcae = GCAE(cfg["num_kp"], 2, 32, cfg["tok_ch"], cfg["seg_len"],
                cfg["n_tokens"])
    tr = TokenTransformer(gcae.token_dim, cfg["nhead"], cfg["num_layers"],
                          cfg["ff_dim"], cfg["n_tokens"], cfg["dropout"])
    return gcae, tr, Shopformer(gcae, tr)


@torch.no_grad()
def segment_scores(model, X: np.ndarray, device, bs: int = 512) -> np.ndarray:
    """Điểm bất thường (MSE) cho mỗi đoạn. X: [N, 2, T, K]."""
    model.eval()
    out = []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i + bs]).to(device)
        err = ((model(xb) - xb) ** 2).mean(dim=(1, 2, 3))
        out.append(err.cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0, dtype=np.float32)


# ---------------------------------------------------------------- dữ liệu
def _runs(frames: np.ndarray):
    """Tách các khoảng khung hình liên tục."""
    if len(frames) == 0:
        return
    start = 0
    for i in range(1, len(frames) + 1):
        if i == len(frames) or frames[i] != frames[i - 1] + 1:
            yield start, i
            start = i


def normalize_segment(seg: np.ndarray) -> np.ndarray:
    """seg: [T, K, 2] -> [2, T, K]. Trừ tâm đoạn, chia chiều cao trung bình."""
    seg = seg - seg.reshape(-1, 2).mean(0)
    height = (seg[..., 1].max(1) - seg[..., 1].min(1)).mean() + 1e-6
    return (seg / height).transpose(2, 0, 1).astype(np.float32)


def make_segments(tracks: dict, seg_len: int, stride: int):
    """tracks: {pid: {frame: kp[K,2]}} -> (X [N,2,T,K], meta [(pid, frames)])."""
    X, meta = [], []
    for pid, fmap in tracks.items():
        frames = np.array(sorted(fmap))
        for a, b in _runs(frames):
            run = frames[a:b]
            for s in range(0, len(run) - seg_len + 1, stride):
                fr = run[s:s + seg_len]
                seg = np.stack([fmap[f] for f in fr])
                X.append(normalize_segment(seg))
                meta.append((pid, fr))
    if not X:
        return np.zeros((0, 2, seg_len, 1), np.float32), meta
    return np.stack(X), meta


def frame_scores(seg_sc: np.ndarray, meta, n_frames: int):
    """Điểm theo khung hình: trung bình theo từng người, lấy max giữa các người."""
    sums, cnts = {}, {}
    for sc, (pid, fr) in zip(seg_sc, meta):
        s = sums.setdefault(pid, np.zeros(n_frames))
        c = cnts.setdefault(pid, np.zeros(n_frames))
        fr = fr[fr < n_frames]
        s[fr] += sc
        c[fr] += 1
    out = np.full(n_frames, np.nan)
    for pid in sums:
        with np.errstate(invalid="ignore", divide="ignore"):
            per = np.where(cnts[pid] > 0, sums[pid] / cnts[pid], np.nan)
        out = np.fmax(out, per)                  # fmax bỏ qua NaN
    return out


def eer_from_roc(fpr, tpr):
    fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[i] + fnr[i]) / 2)


In [ ]:
import importlib; importlib.invalidate_caches()
import shopformer_lite as sf
from shopformer_lite import (add_neck, make_segments, frame_scores, segment_scores,
                             build_models, eer_from_roc)

root = Path(CFG["data_root"])
pkls = sorted(root.rglob("*.pkl"))
npys = {p.stem: p for p in root.rglob("*.npy")}
print(f"{len(pkls)} file .pkl, {len(npys)} file .npy trong {root}")
assert pkls, "Không thấy file .pkl. Kiểm tra CFG['data_root'] (hoặc biến SHOPLIFT_DATA)."

def _first(d, names, default=None):
    for n in names:
        if n in d:
            return d[n]
    return default

def _frame_idx(key):
    try:
        return int(key)
    except (TypeError, ValueError):
        return int(re.findall(r"\d+", str(key))[-1])

def inspect_pkl(path, n=1):
    """In cấu trúc một file .pkl để kiểm tra định dạng."""
    with open(path, "rb") as f:
        d = pickle.load(f)
    print(path.name, "| kiểu:", type(d).__name__, "| số khóa:", len(d))
    non_empty = [k for k in sorted(d.keys(), key=_frame_idx) if d[k]]
    k = non_empty[len(non_empty) // 2] if non_empty else sorted(d.keys(), key=_frame_idx)[0]
    v = d[k]
    print("khung", k, "->", type(v).__name__)
    if isinstance(v, dict):
        for kk, vv in list(v.items())[:2]:
            shape_info = list(vv.keys()) if isinstance(vv, dict) else ([np.shape(x) for x in vv] if isinstance(vv, (list, tuple)) else np.shape(vv))
            print("  ", kk, "->", type(vv).__name__, shape_info)
    elif isinstance(v, (list, tuple)) and v:
        print("  phần tử đầu:", type(v[0]).__name__, (list(v[0].keys()) if isinstance(v[0], dict) else np.shape(v[0])))

def parse_pkl(path):
    """.pkl -> {pid: {frame: keypoints[17,2] theo (x, y)}}. Hỗ trợ định dạng PoseLift [bbox, kp_17x3]."""
    with open(path, "rb") as f:
        d = pickle.load(f)
    tracks = {}
    for key, persons in d.items():
        if not persons:
            continue
        fr = _frame_idx(key)
        if isinstance(persons, dict) and ("keypoints" in persons or "kpts" in persons):
            items = [(_first(persons, ["id", "person_id", "track_id"], 0), persons)]
        elif isinstance(persons, dict):
            items = list(persons.items())
        else:
            items = list(enumerate(persons))
        for k2, info in items:
            if isinstance(info, (list, tuple)) and len(info) >= 2:
                # Định dạng gốc PoseLift Pickle_files: info = [bbox, keypoints_17x3 (y, x, conf)]
                pid = int(k2)
                kp = np.asarray(info[1], dtype=float).reshape(17, -1)[:, [1, 0]]  # đảo (y, x) -> (x, y)
            elif isinstance(info, np.ndarray):
                pid = int(k2)
                kp = np.asarray(info, dtype=float).reshape(17, -1)[:, :2]
            elif isinstance(info, dict):
                pid = _first(info, ["id", "person_id", "track_id", "pid"], k2)
                kp = _first(info, ["keypoints", "keypoint", "kpts", "pose"])
                if kp is None:
                    raise ValueError(f"{path.name}: không thấy khóa keypoints, các khóa có: {list(info.keys())}")
                kp = np.asarray(kp, dtype=float).reshape(17, -1)[:, :2]
            else:
                raise ValueError(f"{path.name}: cấu trúc lạ ở khung {key}. Chạy inspect_pkl() để kiểm tra.")
            tracks.setdefault(pid, {})[fr] = kp
    return tracks

inspect_pkl(pkls[0])

In [ ]:
videos, missing, train_split_from_folder, test_split_from_folder = {}, [], [], []
for p in pkls:
    parent_lower = p.parent.name.lower()
    tracks = parse_pkl(p)
    frames = [f for fm in tracks.values() for f in fm]
    if p.stem in npys:
        labels = np.load(npys[p.stem]).astype(int).ravel()
    elif parent_lower == "train":
        # Trong PoseLift/Pickle_files/Train, toàn bộ 104 video đều là hành vi bình thường (nhãn = 0)
        n_fr = (max(frames) + 1) if frames else 0
        labels = np.zeros(n_fr, dtype=int)
    else:
        missing.append(p.stem); continue
    off = 1 if (frames and min(frames) == 1 and max(frames) == len(labels)) else 0  # chỉ số khung bắt đầu từ 1
    conv = (lambda kp: add_neck(kp)) if CFG["num_kp"] == 18 else (lambda kp: kp)
    tracks = {pid: {f - off: conv(kp) for f, kp in fm.items() if 0 <= f - off < len(labels)}
              for pid, fm in tracks.items()}
    videos[p.stem] = dict(tracks=tracks, labels=labels)
    if parent_lower == "train":
        train_split_from_folder.append(p.stem)
    elif parent_lower == "test":
        test_split_from_folder.append(p.stem)
if missing:
    print("Thiếu file nhãn cho:", missing[:5], "...")

# Chia dữ liệu: ưu tiên chia theo thư mục chuẩn Train/ và Test/ của PoseLift nếu có
rng = random.Random(CFG["seed"])
if train_split_from_folder and test_split_from_folder:
    norm_pool = sorted(train_split_from_folder)
    rng.shuffle(norm_pool)
    n_va = max(1, int(len(norm_pool) * CFG["val_frac"]))
    val_norm, train_norm = norm_pool[:n_va], norm_pool[n_va:]
    test_names = sorted(test_split_from_folder)
    anom = [n for n in test_names if videos[n]["labels"].max() == 1]
    test_norm = [n for n in test_names if n not in anom]
else:
    names = sorted(videos)
    anom = [n for n in names if videos[n]["labels"].max() == 1]
    norm = [n for n in names if n not in anom]
    rng.shuffle(norm)
    n_te = max(1, int(len(norm) * CFG["test_normal_frac"]))
    test_norm, rest = norm[:n_te], norm[n_te:]
    n_va = max(1, int(len(rest) * CFG["val_frac"]))
    val_norm, train_norm = rest[:n_va], rest[n_va:]
    test_names = test_norm + anom
print(f"video: train {len(train_norm)} | val {len(val_norm)} | test {len(test_norm)} bình thường + {len(anom)} có trộm")

def build(names_):
    Xs = []
    for n in names_:
        X, _ = make_segments(videos[n]["tracks"], CFG["seg_len"], CFG["seg_stride"])
        if len(X):
            Xs.append(X)
    return np.concatenate(Xs) if Xs else np.zeros((0, 2, CFG["seg_len"], CFG["num_kp"]), np.float32)

X_train, X_val = build(train_norm), build(val_norm)
print("đoạn khung xương: train", X_train.shape, "| val", X_val.shape)
assert len(X_train) > 0, "Không tạo được đoạn nào. Có thể mỗi người chỉ xuất hiện ngắn hơn seg_len khung."

In [ ]:
# ---- Giai đoạn 1: huấn luyện GCAE (không giám sát, chỉ dữ liệu bình thường)
gcae, tr, model = build_models(CFG)
gcae.to(device); tr.to(device); model.to(device)

def loader(X, shuffle):
    ds = TensorDataset(torch.from_numpy(X))
    return DataLoader(ds, batch_size=CFG["batch"], shuffle=shuffle,
                      drop_last=shuffle and len(X) > CFG["batch"])

@torch.no_grad()
def eval_loss(fn, X):
    if len(X) == 0:
        return float("nan")
    tot = 0.0
    for (xb,) in loader(X, False):
        xb = xb.to(device)
        tot += ((fn(xb) - xb) ** 2).mean().item() * len(xb)
    return tot / len(X)

opt = torch.optim.Adam(gcae.parameters(), lr=CFG["gcae_lr"])
t0 = time.time()
for ep in range(1, CFG["gcae_epochs"] + 1):
    gcae.train(); run = n = 0
    for (xb,) in loader(X_train, True):
        xb = xb.to(device)
        loss = ((gcae(xb) - xb) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        run += loss.item() * len(xb); n += len(xb)
    gcae.eval()
    print(f"[GCAE] epoch {ep:3d} | train {run/n:.5f} | val {eval_loss(gcae, X_val):.5f} | {time.time()-t0:.0f}s")

In [ ]:
# ---- Giai đoạn 2: đóng băng GCAE, huấn luyện Transformer
# Ghi chú: README dùng 10 epoch, lr 5e-5. Với số đoạn ít, 10 epoch có thể chưa đủ hội tụ,
# nên mặc định ở đây là 30 epoch, lr 1e-4. Nếu loss val còn giảm đều, hãy tăng epoch.
for p in gcae.parameters():
    p.requires_grad = False
opt = torch.optim.Adam(tr.parameters(), lr=CFG["tr_lr"])
best, best_state = float("inf"), None
t0 = time.time()
for ep in range(1, CFG["tr_epochs"] + 1):
    tr.train(); gcae.eval()          # GCAE luôn ở chế độ eval (BatchNorm)
    run = n = 0
    for (xb,) in loader(X_train, True):
        xb = xb.to(device)
        loss = ((model(xb) - xb) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        run += loss.item() * len(xb); n += len(xb)
    tr.eval()
    v = eval_loss(model, X_val)
    if v < best:
        best, best_state = v, {k: x.detach().clone() for k, x in tr.state_dict().items()}
    print(f"[Transformer] epoch {ep:3d} | train {run/n:.5f} | val {v:.5f} | {time.time()-t0:.0f}s")
tr.load_state_dict(best_state); tr.eval()
print("val tốt nhất:", round(best, 5))

In [ ]:
# ---- Đánh giá trên video test (frame-level): AUC-ROC, AUC-PR, EER
val_scores = segment_scores(model, X_val, device)
floor = float(val_scores.min()) if len(val_scores) else 0.0

def video_scores(name):
    v = videos[name]
    X, meta = make_segments(v["tracks"], CFG["seg_len"], CFG["seg_stride"])
    sc = segment_scores(model, X, device) if len(X) else np.zeros(0)
    fs = frame_scores(sc, meta, len(v["labels"]))
    fs = np.where(np.isnan(fs), floor, fs)             # khung không có người -> coi là bình thường
    return gaussian_filter1d(fs, CFG["smooth_sigma"]) if CFG["smooth_sigma"] else fs

S, L = [], []
for n in test_names:
    S.append(video_scores(n)); L.append(videos[n]["labels"])
S, L = np.concatenate(S), np.concatenate(L)
assert L.min() == 0 and L.max() == 1, "Tập test cần có cả khung bình thường và khung trộm."

fpr, tpr, _ = roc_curve(L, S)
prec, rec, _ = precision_recall_curve(L, S)
metrics = dict(auc_roc=float(roc_auc_score(L, S)) * 100,
               auc_pr=float(auc(rec, prec)) * 100,
               eer=eer_from_roc(fpr, tpr),
               test_frames=int(len(L)), test_anomalous_frames=int(L.sum()))
print(json.dumps(metrics, indent=2))
print("Bài báo (PoseLift, mã chính thức): AUC-ROC 69.15 | AUC-PR 44.49 | EER 0.38")

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3.2))
plt.hist(S[L == 0], bins=50, alpha=.6, density=True, label="bình thường")
plt.hist(S[L == 1], bins=50, alpha=.6, density=True, label="trộm")
plt.xlabel("điểm bất thường"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ---- Chọn ngưỡng từ điểm các đoạn bình thường (val) và lưu "model bundle"
threshold = float(np.percentile(val_scores, CFG["threshold_pct"])) if len(val_scores) else None
X_test_norm = build(test_norm)
test_norm_sc = segment_scores(model, X_test_norm, device)
fp_rate = float((test_norm_sc > threshold).mean()) if len(test_norm_sc) and threshold is not None else None
print(f"ngưỡng (phân vị {CFG['threshold_pct']}): {threshold} | tỉ lệ đoạn bình thường vượt ngưỡng trên test: {fp_rate}")

out = Path(CFG["out_dir"]); out.mkdir(parents=True, exist_ok=True)
torch.save(dict(gcae=gcae.state_dict(), transformer=tr.state_dict(), cfg=CFG), out / "model.pt")
json.dump(dict(seg_len=CFG["seg_len"], seg_stride=CFG["seg_stride"], num_kp=CFG["num_kp"],
               keypoints="COCO17 + neck (trung điểm hai vai)" if CFG["num_kp"] == 18 else "COCO17",
               fps=15, normalization="trừ tâm đoạn; chia chiều cao trung bình của khung xương"),
          open(out / "preprocess.json", "w"), indent=2, ensure_ascii=False)
json.dump(dict(threshold=threshold, percentile=CFG["threshold_pct"], score="MSE tái tạo theo đoạn",
               normal_segment_exceed_rate_on_test=fp_rate),
          open(out / "calibration.json", "w"), indent=2)
json.dump(dict(model="shopformer-lite (cài đặt lại, không phải mã chính thức)", dataset="PoseLift",
               metrics=metrics, videos=dict(train=len(train_norm), val=len(val_norm),
               test_normal=len(test_norm), test_anomalous=len(anom)), trained_at=time.strftime("%Y-%m-%d %H:%M")),
          open(out / "meta.json", "w"), indent=2, ensure_ascii=False)
shutil.copy("shopformer_lite.py", out / "shopformer_lite.py")
print("Đã lưu bundle vào", out.resolve())

## Dùng lại model đã train

```python
import json, torch
from shopformer_lite import build_models
b = torch.load("checkpoints/shopformer_lite_v1/model.pt", map_location="cpu")
gcae, tr, model = build_models(b["cfg"])
gcae.load_state_dict(b["gcae"]); tr.load_state_dict(b["transformer"]); model.eval()
thr = json.load(open("checkpoints/shopformer_lite_v1/calibration.json"))["threshold"]
```

Khi suy luận phải dùng **đúng** tiền xử lý đã ghi trong `preprocess.json` (15 FPS, đoạn 24 khung, thêm điểm cổ, chuẩn hóa như hàm `normalize_segment`).
Để đưa vào dự án, chép `shopformer_lite.py` vào `src/shoplift_iot/model/`.